In [3]:
import sqlite3
import pandas as pd
from lifelines import CoxPHFitter

In [4]:
con = sqlite3.connect('../data/playerpulse.db')
m = pd.read_sql('SELECT account_id, MIN(start_time) f, MAX(start_time) l, '
                'AVG(win) win_rate, COUNT(*) * 1.0 n FROM matches GROUP BY account_id', con)
t_max = con.execute('SELECT MAX(start_time) FROM matches').fetchone()[0]
m['days'] = (m.l - m.f) / 86400 + 1
m['churned'] = (m.l < t_max - 30 * 86400).astype(int)
m['matches_per_day'] = m.n / m.days
cph = CoxPHFitter().fit(m[['days', 'churned', 'win_rate', 'matches_per_day']], 'days', 'churned')
cph.print_summary()

<lifelines.CoxPHFitter: fitted with 259 total observations, 197 right-censored observations>
             duration col = 'days'
                event col = 'churned'
      baseline estimation = breslow
   number of observations = 259
number of events observed = 62
   partial log-likelihood = -308.48
         time fit was run = 2026-11-08 10:37:13 UTC

---
                 coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                        
win_rate        -4.31      0.01      0.81           -5.90           -2.71                0.00                0.07
matches_per_day -0.22      0.80      0.18           -0.58            0.14                0.56                1.15

                 cmp to     z      p  -log2(p)
covariate                                     
win_rate           0.00 -5.30 <0.005     23.01
matches_per_day    0.00 -1.20   0.23      2.13
---
Concordance = 0.67
Partial AIC = 620.96
log-likelihood ratio test = 28.29 on 2 df
-log2(p) of ll-ratio test = 20.41